In [ ]:
from google.colab import drive
import os

# This will prompt you to authorize access to your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create the local workspace directory
!mkdir -p dataset

# Unpack it instantly from your Drive directly into the local Colab disk
!tar -xzf "/content/drive/MyDrive/crop_dataset.tar.gz" -C dataset

Streaming output truncated to the last 5000 lines.
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import os
import glob
import torch
import json
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# 1. Detect Cloud NVIDIA GPU Acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using cloud acceleration device: {device}")

model_id = "IDEA-Research/grounding-dino-base"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

IMAGE_DIR = "dataset/crop"
OUTPUT_DIR = "dataset/labels"
JSON_DIR = "dataset/json_labels"
# We skip ANNOTATED_DIR entirely to eliminate image drawing bottlenecks!

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)

CLASSES = ["cat"]
TEXT_PROMPT = "a cat ."
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

def convert_to_yolo(box, img_w, img_h):
    xmin, ymin, xmax, ymax = box
    box_w = xmax - xmin
    box_h = ymax - ymin
    return [
        (xmin + (box_w / 2)) / img_w,
        (ymin + (box_h / 2)) / img_h,
        box_w / img_w,
        box_h / img_h
    ]

# Gather all paths
image_extensions = ("*.jpg", "*.jpeg", "*.png", "*.webp")
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, ext)))

# RESUME LOGIC: Filter out images that already have a generated label file
image_paths = [
    p for p in image_paths
    if not os.path.exists(os.path.join(OUTPUT_DIR, f"{os.path.splitext(os.path.basename(p))[0]}.txt"))
]

print(f"Resuming pipeline. Processing remaining {len(image_paths)} images...")

# 2. Execution Loop
for img_path in tqdm(image_paths, desc="GPU Auto-Labeling"):
    try:
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        inputs = processor(images=image, text=TEXT_PROMPT, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            target_sizes=[(h, w)]
        )[0]

        yolo_lines = []
        json_data = {"image": os.path.basename(img_path), "width": w, "height": h, "annotations": []}

        labels_key = "text_labels" if "text_labels" in results else "labels"

        for score, label, box in zip(results["scores"], results[labels_key], results["boxes"]):
            if isinstance(label, int):
                cleaned_label = CLASSES[label] if label < len(CLASSES) else "cat"
            else:
                cleaned_label = label.replace("a ", "").strip()

            if cleaned_label in CLASSES:
                class_id = CLASSES.index(cleaned_label)
                box_list = box.tolist()
                confidence = float(score)

                yolo_box = convert_to_yolo(box_list, w, h)
                yolo_lines.append(f"{class_id} " + " ".join([f"{coord:.6f}" for coord in yolo_box]))

                json_data["annotations"].append({
                    "class": cleaned_label,
                    "confidence": confidence,
                    "bbox_xyxy": box_list
                })

        base_name = os.path.splitext(os.path.basename(img_path))[0]

        # Save output YOLO .txt string
        with open(os.path.join(OUTPUT_DIR, f"{base_name}.txt"), "w") as f:
            f.write("\n".join(yolo_lines))

        # Save raw JSON metadata
        with open(os.path.join(JSON_DIR, f"{base_name}.json"), "w") as f:
            json.dump(json_data, f, indent=4)

    except Exception as e:
        print(f"\nSkipping {img_path} due to error: {e}")

print("\nProcessing complete!")

Using cloud acceleration device: cuda


Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

Resuming pipeline. Processing remaining 5643 images...


GPU Auto-Labeling: 100%|██████████| 5643/5643 [1:10:18<00:00,  1.34it/s]


Processing complete!


In [ ]:
!tar -czf grounding_dino_labels.tar.gz dataset/labels dataset/json_labels dataset/annotated